# BigSMILES and BigSmirk

<a href="https://colab.research.google.com/github/BattModels/smirk/blob/main/docs/big_smirk_demo.ipynb">
    <img alt="Open In Colab" src="https://colab.research.google.com/assets/colab-badge.svg">
</a>
<a href="https://mybinder.org/v2/gh/BattModels/smirk/main?urlpath=%2Fdoc%2Ftree%2Fdocs%2Fbig_smirk_demo.ipynb">
    <img alt="Binder" src="https://mybinder.org/badge_logo.svg">
</a>



BigSmirk tokenizes the [BigSMILES] encoding for macromolecules all the way down to their constituent elements.

Let's see it in action!

[BigSMILES]: https://olsenlabmit.github.io/BigSMILES/docs/#the-bigsmiles-project

🐍 Installation is easy with pre-build binaries on [PyPI](https://pypi.org/project/smirk/) and [GitHub](https://github.com/BattModels/smirk/releases). Just run: `pip install smirk`

> Installing from source? See [installing from source](./developer.md#installing-from-source) for instructions.

In [ ]:
!python -m pip install smirk transformers

## First steps

🤗 smirk subclasses Hugging Face's [PreTrainedTokenizerBase](#transformers.PreTrainedTokenizerBase) for seamless compatibility and leverages [Tokenizers] for raw rust-powered speed. No need to learn another framework; everything works out of the box 🎁

[Tokenizers]: https://huggingface.co/docs/tokenizers/index

In [ ]:
from smirk import SmirkBigSmilesFast

# Just import and tokenize!
bigsmirk = SmirkBigSmilesFast()
bigsmirk("{[][$]CC[$],[$]CC(CC)[$][]}") # ethylene butene copolymer

In [ ]:
# Batch Tokenization with Padding
batch = bigsmirk([
    "[H]O{[>][<]C(=O)CCCCC(=O)[<],[>]NCCCCCCN[>][<]}[H]", # nylon-6,6
    "{[][<]OCC[>][<]}{[>][<]OC(C)C[>][]}", # block copolymer
    "{[][<]C(=O)c1ccc(cc1)C(=O)[<],[>]OCCO[>][]}", # alternation co-polymer
], padding="longest")
batch

In [ ]:
# Back to polymers!
bigsmirk.batch_decode(batch["input_ids"], skip_special_tokens=True)

### Token Coloring Render

Visualize BigSMILES token boundaries for PVC (Polyvinyl chloride ) and sPP (Syndiotactic Polypropylene) by coloring each token in sequence.

In [ ]:
import hashlib
from html import escape
from IPython.display import HTML


def render_colored_tokens(text, tokenizer=bigsmirk):
    tokens = tokenizer.tokenize(text)
    palette = [
        "#f94144", "#f3722c", "#f8961e", "#f9844a", "#f9c74f",
        "#90be6d", "#43aa8b", "#4d908e", "#577590", "#277da1",
    ]
    spans = []
    for tok in tokens:
        digest = hashlib.sha1(tok.encode("utf-8")).digest()
        color = palette[int.from_bytes(digest[:2], "big") % len(palette)]
        label = escape(tok)
        spans.append(
            f"<span style='display:inline-block;margin:2px;padding:3px 6px;border-radius:6px;background:{color};color:#111;font-family:monospace'>{label}</span>"
        )
    return HTML("<div style='line-height:2.2'>" + "".join(spans) + "</div>")


pvc = "{[][$]CC(Cl)[$][]}"
render_colored_tokens(pvc)

In [ ]:
spp = "CC{[>][<]C[C@@H](C)C[C@H](C)[>];[<]C=CC,[<]C[C@H](C)C=CC[]}"
render_colored_tokens(spp)

## Zero to Polymer Foundation Model with Smirk!

Let's train a small [RoBERTa] model on polymers from [S. Choi et al., 2024] using Hugging Face and smirk.

[RoBERTa]: https://doi.org/10.48550/ARXIV.1907.11692
[S. Choi et al., 2024]:https://www.nature.com/articles/s41597-024-03212-4

In [ ]:
!python -m pip install accelerate datasets torch

### Dataset Preprocessing

Download the dataset for generated by [Choi et al] from [figshare] and unzip it:

```
curl -L -o with_Tg.zip "https://springernature.figshare.com/ndownloader/files/42507037"
unzip with_Tg.zip -d with_Tg
```

[Choi et al]: https://www.nature.com/articles/s41597-024-03212-4#citeas
[figshare]: "https://springernature.figshare.com/ndownloader/files/42507037"

In [ ]:
from datasets import load_dataset
import os

# Location to unzipped data
data_dir =  "with_Tg"
dataset=load_dataset("csv", data_files=[os.path.join(data_dir,"JCIM_sup_bigsmiles.csv")])["train"].select_columns(["BigSMILES"]).train_test_split(test_size=0.2)
dataset=dataset.map(bigsmirk, input_columns=["BigSMILES"], desc="Tokenizing")

> 💡 Hugging Face/ Tokenizers may raise a warning about being forked as we've already used our tokenizers (this isn't a smirk issue).
> It's harmless, but when actually training it's best to avoid tokenization until after the fork to benefit from the rust-level parallelism

🎉 That's it! We've tokenized all of the BigSMILES dataset using smirk!

In [ ]:
dataset["train"].to_pandas().head()

### Training
Once we've tokenized the dataset, training the model is just a matter of configuration.

In [ ]:
from transformers import Trainer, RobertaForMaskedLM, RobertaConfig, DataCollatorForLanguageModeling

# A very small model for demonstrating training a molecular foundation model with smirk 
config = RobertaConfig(
    vocab_size=len(bigsmirk),
    hidden_size=256,
    intermediate_size=1024,
    num_hidden_layers=4,
    num_attention_heads=4,
)
model = RobertaForMaskedLM(config)

# Setup up the trainer to use our dataset
trainer = Trainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=bigsmirk,
    data_collator=DataCollatorForLanguageModeling(bigsmirk), # The data collator needs to know about our tokenizer
)

In [ ]:
trainer.train()

### MLM Example: Predict a Masked Token

Mask one token in a BigSMILES string and ask the trained model for top predictions.

In [ ]:
import torch

inference_model = trainer.model
inference_model.eval()
device = next(inference_model.parameters()).device

sample = dataset["test"][5]["BigSMILES"]

# Encode and choose a position to mask
encoded = bigsmirk(sample, add_special_tokens=False)
input_ids = encoded["input_ids"]
tokens = bigsmirk.convert_ids_to_tokens(input_ids)
mask_pos = len(tokens) // 2

masked_ids = input_ids.copy()
masked_ids[mask_pos] = bigsmirk.mask_token_id
masked_tokens = bigsmirk.convert_ids_to_tokens(masked_ids)

inputs = {
    "input_ids": torch.tensor([masked_ids],device=device),
    "attention_mask": torch.ones((1, len(masked_ids)),  device=device),
}

with torch.no_grad():
    logits = inference_model(**inputs).logits[0, mask_pos].detach().cpu()
    probs = torch.softmax(logits, dim=-1)

top_k = 5
top_ids = torch.topk(probs, k=top_k).indices.tolist()
top_tokens = bigsmirk.convert_ids_to_tokens(top_ids)

print("Original:", sample)
print("Masked :", "".join(masked_tokens))
print(f"Masked token index: {mask_pos} (original token: {tokens[mask_pos]})")
print("\nTop predictions:")

for rank, (tok_id, tok) in enumerate(zip(top_ids, top_tokens), start=1):
    candidate_ids = masked_ids.copy()
    candidate_ids[mask_pos] = tok_id
    candidate = bigsmirk.decode(candidate_ids, skip_special_tokens=True)
    score = probs[tok_id].item()
    print(f"{rank}. token={tok!r:>4}  p={score:.4f}  ->  {candidate}")